In [1]:
import sys

import pm4py

import pandas as pd
import numpy as np

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts
from model.next_event_model import ProcessLSTM, train_ProcessLSTM, validate_ProcessLSTM

### --- Preprocess dataset ---

In [2]:
set_seed(seed=42)

In [3]:
log = pm4py.read_xes("../../data/Hospital Billing - Event Log.xes")

C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(
C:\Users\dcoralage\Downloads\counterfactual_prediction_experiments\counterfactual_env\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/100000 [00:00<?, ?it/s]

In [4]:
df = pm4py.convert_to_dataframe(log)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451359 entries, 0 to 451358
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   isCancelled           108222 non-null  object             
 1   diagnosis             81885 non-null   object             
 2   time:timestamp        451359 non-null  datetime64[ns, UTC]
 3   caseType              101626 non-null  object             
 4   speciality            100000 non-null  object             
 5   org:resource          249081 non-null  object             
 6   concept:name          451359 non-null  object             
 7   blocked               100000 non-null  object             
 8   isClosed              100390 non-null  object             
 9   flagD                 100038 non-null  object             
 10  flagB                 100000 non-null  object             
 11  flagA                 100000 non-null  object       

In [6]:
df.isnull().any()

isCancelled              True
diagnosis                True
time:timestamp          False
caseType                 True
speciality               True
org:resource             True
concept:name            False
blocked                  True
isClosed                 True
flagD                    True
flagB                    True
flagA                    True
state                    True
lifecycle:transition    False
case:concept:name       False
closeCode                True
actRed                   True
actOrange                True
flagC                    True
msgCount                 True
version                  True
msgType                  True
msgCode                  True
dtype: bool

In [7]:
event_cols = [
    'org:resource', 'diagnosis', 'closeCode',
    "isCancelled",
    "isClosed",
    "caseType",
    "speciality",
    "blocked",
    "flagA",
    "flagB",
    "flagC",
    "flagD",
    "state",
    "actRed",
    "actOrange",
    "version",
    "msgType",
    "msgCode",
    "msgCount"
]

delete_cols = []

for col in event_cols:
    s = df[col].replace("", pd.NA)

    missing = s.isna().mean()
    max_freq = s.value_counts(normalize=True, dropna=True).max()

    print(
        f"{col:15} "
        f"missing={missing:.1%} "
        f"largest_class={max_freq:.1%}"
    )

    if missing >= 0.80:
        delete_cols.append(col)

print("\nColumns to delete:")
print(delete_cols)

org:resource    missing=44.8% largest_class=29.2%
diagnosis       missing=81.9% largest_class=2.5%
closeCode       missing=84.1% largest_class=34.7%
isCancelled     missing=76.0% largest_class=92.4%
isClosed        missing=77.8% largest_class=92.3%
caseType        missing=77.5% largest_class=41.8%
speciality      missing=77.8% largest_class=22.6%
blocked         missing=77.8% largest_class=100.0%
flagA           missing=77.8% largest_class=100.0%
flagB           missing=77.8% largest_class=100.0%
flagC           missing=85.3% largest_class=95.5%
flagD           missing=77.8% largest_class=70.2%
state           missing=15.9% largest_class=41.9%
actRed          missing=85.3% largest_class=95.5%
actOrange       missing=85.3% largest_class=99.8%
version         missing=84.9% largest_class=38.5%
msgType         missing=99.6% largest_class=87.3%
msgCode         missing=99.6% largest_class=83.2%
msgCount        missing=85.2% largest_class=97.6%

Columns to delete:
['diagnosis', 'closeCode', '

In [8]:
df = df.drop(columns=['diagnosis', 'closeCode', 'flagC', 'actRed', 'actOrange', 'version', 'msgType', 'msgCode', 'msgCount', 'org:resource'])

In [9]:
df['case:concept:name'] = df['case:concept:name'].astype('string')
df['concept:name'] = df['concept:name'].astype('string')
df['lifecycle:transition'] = df['lifecycle:transition'].astype('string')

df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], errors='coerce')

df['isCancelled'] = df['isCancelled'].astype('string')
df['isClosed'] = df['isClosed'].astype('string')
df['caseType'] = df['caseType'].astype('string')
df['speciality'] = df['speciality'].astype('string')
df['blocked'] = df['blocked'].astype('string')
df['flagD'] = df['flagD'].astype('string')
df['flagB'] = df['flagB'].astype('string')
df['flagA'] = df['flagA'].astype('string')
df['state'] = df['state'].astype('string')

In [10]:
df = df.sort_values(by=['case:concept:name', 'time:timestamp'], ascending=[True, True])

In [11]:
df['time_delta'] = df.groupby('case:concept:name')['time:timestamp'].diff()
df['time_delta'] = df['time_delta'].dt.total_seconds()
df['time_delta'] = df['time_delta'].fillna(0)

In [12]:
exclude_cols = ["case:concept:name", "time:timestamp"]

sorted_cols = sorted(
    [c for c in df.columns if c not in exclude_cols]
)

df = df[exclude_cols + sorted_cols]

In [13]:
df.head(20)

,case:concept:name,time:timestamp,blocked,caseType,concept:name,flagA,flagB,flagD,isCancelled,isClosed,lifecycle:transition,speciality,state,time_delta
0,A,2012-12-16 19:33:10+00:00,False,A,NEW,False,False,True,False,True,complete,A,In progress,0.0
1,A,2013-12-15 19:00:37+00:00,<NA>,<NA>,FIN,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,Closed,31447647.0
2,A,2013-12-16 03:53:38+00:00,<NA>,<NA>,RELEASE,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,Released,31981.0
3,A,2013-12-17 12:56:29+00:00,<NA>,<NA>,CODE OK,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,<NA>,118971.0
4,A,2013-12-19 03:44:31+00:00,<NA>,<NA>,BILLED,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,Billed,139682.0
146,AA,2012-12-26 08:50:18+00:00,False,B,NEW,False,False,True,False,True,complete,L,In progress,0.0
147,AA,2012-12-26 08:50:59+00:00,<NA>,<NA>,CHANGE DIAGN,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,In progress,41.0
148,AA,2013-02-14 21:06:33+00:00,<NA>,<NA>,FIN,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,Closed,4364134.0
149,AA,2013-02-14 22:12:10+00:00,<NA>,<NA>,RELEASE,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,Released,3937.0
150,AA,2013-02-18 01:44:10+00:00,<NA>,<NA>,CODE OK,<NA>,<NA>,<NA>,<NA>,<NA>,complete,<NA>,<NA>,271920.0


In [14]:
num_cases = df['case:concept:name'].nunique()
print(f"Total number of unique cases: {num_cases}")

Total number of unique cases: 100000


### --- Feature Configurations ---

In [15]:
# --- Define feature specs ---
feature_specs = {

    "isCancelled": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           True
    },

    "isClosed": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           True
    },

    "caseType": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "speciality": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "blocked": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "flagD": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "flagB": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "flagA": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },

    "state": {
        "type":            "categorical",
        "level":           "event",
        "vary":            True,
    },
   
    "time_delta": {
        "type":            "continuous",
        "level":           "event",
        "vary":            True,
        "quantile_low":    0.20,
        "quantile_high":   0.80, 
    },

    # immutable
    "concept:name": {
        "type":            "categorical", 
        "level":           "event",
        "vary":            False
    },

    "lifecycle:transition": {
        "type":           "categorical", 
        "level":          "event",
        "vary":           False
    },
}

In [16]:
feature_config = FeatureConfig.from_dataframe(
    df=df,
    feature_specs=feature_specs,
    activity_feature="concept:name",
    is_robust=True,
    default_quantile_low=0.05,
    default_quantile_high=0.95
)

feature_config.save()

In [17]:
# feature_config = FeatureConfig.load()

In [18]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['blocked', 'caseType', 'concept:name', 'flagA', 'flagB', 'flagD', 'isCancelled', 'isClosed', 'lifecycle:transition', 'speciality', 'state', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
isCancelled                    categorical    event    yes    ['False', 'True']                        N/A        data_derived        
isClosed                       categorical    event    yes    ['False', 'True']                        N/A        data_derived        
caseType                       categorical    event    yes    ['A', 'B', 'C', ...]                     N/A     

### --- Next event prediction model ---

In [19]:
cat_cols = df.select_dtypes(include=["string"]).columns
for col in cat_cols:
    df[col] = df[col].fillna("NA").astype('string')

In [20]:
case_ids = df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = df[df["case:concept:name"].isin(train_cases)].copy()
val_df   = df[df["case:concept:name"].isin(val_cases)].copy()

In [21]:
preprocessor_artifacts = PreprocessorArtifacts.build(
    df=train_df,
    feature_config=feature_config,      
    scaler_type="robust",
)

preprocessor_artifacts.save()

In [22]:
# preprocessor_artifacts = PreprocessorArtifacts.load()

In [23]:
preprocessor_artifacts.summary()

===================PreprocessorArtifacts====================
  scaler:              RobustScaler
  encoders:            ['isCancelled', 'isClosed', 'caseType', 'speciality', 'blocked', 'flagD', 'flagB', 'flagA', 'state', 'concept:name', 'lifecycle:transition']
  activity_prototypes: 18 activities


In [24]:
# Transform nan cols to 0
float_cols = df.select_dtypes(include=["float32", "float64"]).columns
df[float_cols] = df[float_cols].fillna(0)
train_df[float_cols] = train_df[float_cols].fillna(0)
val_df[float_cols] = val_df[float_cols].fillna(0)

In [25]:
train_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=train_df,
    sort_field="time:timestamp"
)

val_dataset = preprocessor_artifacts.transform_dataframe_to_processdataset(
    case_id_field="case:concept:name", 
    df=val_df,
    sort_field="time:timestamp"
)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
print(preprocessor_artifacts.get_categorical_feature_cardinality())

{'dynamic_categorical_info': {'blocked': 3, 'caseType': 16, 'concept:name': 18, 'flagA': 2, 'flagB': 2, 'flagD': 3, 'isCancelled': 3, 'isClosed': 3, 'lifecycle:transition': 1, 'speciality': 24, 'state': 11}, 'static_categorical_info': {}}


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [29]:
device

device(type='cuda')

In [30]:
criterion = torch.nn.CrossEntropyLoss()

In [31]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/hospital_billing-model_output.txt")

Epoch 020/100 | Train Loss: 0.1447 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.1310 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.1229 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.1163 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.1135 | LR: 1.00e-06
Time taken for next event model (training): 8929.816230 seconds
Time taken for next event model (validation): 5.939082 seconds
Val loss: {'loss': 0.23534496577375455, 'accuracy': 0.9558544822377703, 'f1_macro': 0.7053109167036091, 'f1_weighted': 0.9531228372676634}


In [32]:
embedding_metadata = preprocessor_artifacts.get_embedding_metadata()

model = ProcessLSTM(
    dynamic_categorical_info=embedding_metadata["dynamic_categorical_info"],
    static_categorical_info=embedding_metadata["static_categorical_info"],
    n_dynamic_continuous=embedding_metadata["n_dynamic_continuous"],
    n_static_continuous=embedding_metadata["n_static_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ProcessLSTM(
    model=model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

model.save()

In [33]:
# model = ProcessLSTM.load()

In [34]:
val_loss = validate_ProcessLSTM(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [35]:
if df["time:timestamp"].dt.tz is not None:
    df["time:timestamp"] = df["time:timestamp"].dt.tz_convert(None)
df.to_excel("../../data/hospital_billing.xlsx", index=False, engine="openpyxl")

In [36]:
sys.stdout = original_stdout
log_file.close()